# [2.2: word2vec] // Word Embeddings

TF–IDF represents documents as vectors based on **word overlap**. This works well when similar documents use the *same words*, but it fails when similar ideas are expressed using **different vocabulary**. For example, with TF–IDF, the words *“dog”* and *“cat”* are treated as completely unrelated unless they co-occur. so, the TF-IDF vectors of articles conatinaing these words might be very dissimilar, even though the articles might be clearly related.

To overcome this limitation, we need a representation that captures **semantic similarity between words**, not just exact matches. This is where **word embeddings** come in.

## Word2Vec: Words as Vectors with Meaning

**Word2Vec** is a technique that represents words as dense numerical vectors in such a way that **semantically similar words end having similar vectors**.

We're not going into detail as to how these word vectors are computed, but the core idea is simple: We look at the context in which words are used (so the words that typically surround it in a sentence. **Words that appear in similar contexts tend to have similar meanings.**

For example, the words *“dog”* and *“cat”* might occur in similar contexts (surrounded by words like "pet", "animal", "owner", "feed", "cute")), This means that the vectors representing those words will be very similar.

spaCy provides **pretrained Word2Vec-style embeddings**, trained on very large text corpora, which we can use directly without training our own models. So we don't have to compute these ourselves. We can use them as is.

> #### Relation to LLM's
> 
> Word2Vec was one of the first widely successful embedding models and laid the groundwork for modern NLP. Today’s large language models (LLMs), such as chatGPT, are built on the same fundamental idea:
> * represent language as vectors,
> * where distance and direction encode meaning.
> Modern models go much further: they produce **context-dependent embeddings** (the meaning of a word depends on the sentence), but Word2Vec is an important first step capturing semantics numerically.

## Using Word Embeddings for Recommender Systems

For recommender systems, word embeddings allow us to build content-based recommenders that generalize better than TF–IDF.

A common approach is:

1. represent each document by combining the embeddings of its words (e.g., averaging),
2. compute cosine similarity between document vectors,
3. use kNN (as before) to generate recommendations.

In the next notebook, we will focus on the first two steps. We will not build a full recommender system again, since the recommendation logic itself should now be familiar.

# Getting started

Start by loading the necessary libraries.

In [ ]:
import pandas as pd
import numpy as np
import spacy
import pooch

%reload_ext autoreload
%autoreload 2


DATA_REPO = "https://raw.githubusercontent.com/uvapl/recommender-systems/2025/data/m2/"
for fname in ["MIND-micro-news.tsv"]:
    pooch.retrieve(url = DATA_REPO + fname, known_hash=None, fname=fname, path="data", progressbar=True)
for fname in ["tests_m2.py"]:
    pooch.retrieve(url = DATA_REPO + fname, known_hash=None, fname=fname, path=".", progressbar=True)

import tests_m2

## Load the spaCy Model

This time, we need the **medium-sized spaCy model** (`_md`) instead of the small model (`_sm`).
The small model does **not** include pretrained word embeddings, while the medium model does.

In [ ]:
# not needed (copy and run in other cell if download above didn't work)
# from spacy.cli import download 
# download("en_core_web_sm")

nlp = spacy.load('en_core_web_md')

# Similarities

Have a look at the code below.

You can see how spaCy’s `.similarity()` function can be used to compute the similarity between two pieces of text. In particular, you will notice that *“dog”* and *“cat”* are much more similar to each other than *“dog”* and *“car”*.

This is because, under the hood, spaCy uses **word embeddings** to compute these similarities. Rather than relying on exact word overlap (as TF–IDF does), it uses the **semantic information** encoded in the embeddings. Since *“dog”* and *“cat”* are often used in similar contexts, their vectors are close together in the embedding space, resulting in a high similarity score.

In [ ]:
dog_cat_similarity = 0
dog_car_similarity = 0

dog = nlp("dog")
cat = nlp("cat")
car = nlp("car")

dog_cat_similarity = dog.similarity(cat)
dog_car_similarity = dog.similarity(car)

print(f"Similarity between `dog` and `cat`: {dog_cat_similarity:.3f}")
print(f"Similarity between `dog` and `car`: {dog_car_similarity:.3f}")

## Vectors

Let’s take a look at what an embedding vector actually looks like. You can inspect it using the `.vector` attribute, as shown in the cell below.

On its own, this vector is not very meaningful. It is simply a list of 300 numbers that appear arbitrary. The meaning of an embedding vector only emerges **in relation to other vectors**: distances between vectors encode semantic relationships.

In [ ]:
print(cat.vector)

## Cosine Similarity

spaCy computes text similarity by taking the **cosine similarity** between the underlying embedding vectors. We can verify this by explicitly computing the cosine similarity ourselves.

### Question 1

*3 pts.*

Implement (or copy and adapt from earlier notebooks) the `cosine_similarity()` function below.
The input should be **two Pandas Series** (these were converted from NumPy arrays for you, so you can continue working with Pandas as before). The function should return a **single float** representing the cosine similarity between the two vectors.

If everything is implemented correctly, your result should match the similarity score produced by spaCy, above.

In [ ]:
def cosine_similarity(vector1: pd.Series, vector2: pd.Series) -> float:
    # your code here

dog_cat_similarity_cos = cosine_similarity(pd.Series(dog.vector), pd.Series(cat.vector))
dog_car_similarity_cos = cosine_similarity(pd.Series(dog.vector), pd.Series(car.vector))

print(f"Similarity between `dog` and `cat`: {dog_cat_similarity_cos:.3f}")
print(f"Similarity betwwen `dog` and `car`: {dog_car_similarity_cos:.3f}")

# Texts of Multiple Words

spaCy is not limited to comparing individual words. It can also compute similarities between **longer pieces of text**, such as phrases, sentences, or entire documents.

When you call `.similarity()` on multi-word texts, spaCy combines the word embeddings of the individual tokens (roughly by averaging them) to create a single vector representation for the whole text. It then computes the cosine similarity between those text-level vectors in the same way as before.

This allows us to compare not just words, but entire articles or summaries based on their **semantic content**, even when they do not share exact vocabulary.

In [ ]:
dog_house = nlp("dog house")
raven = nlp("Once upon a midnight dreary, while I pondered, weak and weary")
print(dog_house.similarity(raven))

### Question 2

*2 pts.*

spaCy computes vector representations for multi-word texts by taking the **mean of the embedding vectors of the individual tokens**. You can verify this behavior yourself.

Below, you are provided with the embedding vectors for **"dog house"**, **"dog"**, and **"house"**. Use the `cosine_similarity()` function from the previous question to show that the vector for *"dog house"* is equal (or extremely close) to the **mean of the vectors** for *"dog"* and *"house"*.

In [ ]:
dog_house = nlp("dog house")
dog_house1_vect = pd.Series(dog_house.vector)

dog = nlp("dog")
house = nlp("house")
dog_vect = pd.Series(dog.vector)
house_vect = pd.Series(house.vector)

# your code here

# Using Word2Vec for MIND

As you have seen, spaCy makes it straightforward to use Word2Vec-style embeddings to compute semantic similarities between texts. We can use this to build an improved **similarity matrix** compared to TF–IDF, because embeddings can capture related meanings even when the exact words differ.

In the next step, reload the news data below so we can recompute document vectors and similarities using embeddings.

In [ ]:
# Read data
# Column names (see MIND documentation)
columns_articles = [
    "NewsID",          # unique ID of the news article
    "Category",        # coarse-grained topic (e.g., Sports)
    "SubCategory",     # fine-grained topic (e.g., Football)
    "Title",           # news headline
    "Abstract",        # short summary
]
news_df = pd.read_csv("data/MIND-micro-news.tsv", sep="\t", index_col = "NewsID", header=None, names=columns_articles, encoding='utf-8')[["Title", "Abstract"]]

display(news_df.head())

### Question 3

*4 pts.*

Complete the function `cosine_similarity_matrix()` below.
The input is a Pandas `Series` containing the combined text (title + abstract) for each news article.

Your function should compute a **cosine similarity matrix** between all news items using spaCy’s `.similarity()` method, and return the result as a Pandas DataFrame indexed by the article IDs.

In [ ]:
def cosine_similarity_matrix(text_series: pd.Series) -> pd.DataFrame:
    # your code here


similarity = cosine_similarity_matrix(news_df["Title"] + " " + news_df["Abstract"])
display(similarity.style.format(precision=2).background_gradient())

In [ ]:
# test your solution

tests_m2.w2v_test_03(similarity)

# Question 4
*2 pts.*

You see that the similarity scores for items that scored quite low using TF-IDF, now ofte score much hight. Why do you think that is? Argue below. 

YOUR ANSWER HERE

## Conclusion

In this notebook, we explored **word embeddings** as an alternative to TF–IDF for representing textual content. Unlike TF–IDF, which relies on exact word overlap, Word2Vec captures **semantic relationships between words**, allowing texts with different vocabularies but similar meanings to be recognized as related.

Training your own Word2Vec vectors is much more involved than creating your own TF-IDF vectors, but fortunately, spaCy provides us with pretrained vectors, allowing us to compute meaningful similarities between documents with very little additional effort. 